# Hubmicroo Voice Assistant — Kaggle Test Environment
> **Testing only** — sessions expire after 12 h. For production use `docker compose up`.

Enable: **GPU T4 x2** + **Internet** in Kaggle Settings before running.

In [ ]:
# ── Step 1: System dependencies ───────────────────────────────────────────
import subprocess, sys

def run(cmd, **kw):
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True, **kw)
    if result.returncode not in (0, 5):  # 5 = systemd harmless
        print(f'WARNING: exit {result.returncode}')
    return result

run('apt-get update -qq && apt-get install -y --no-install-recommends ffmpeg curl wget tar')

In [ ]:
# ── Step 2: Python packages ────────────────────────────────────────────────
run('pip install -q fastapi uvicorn[standard] pydantic pydantic-settings python-multipart')
run('pip install -q python-jose[cryptography] passlib[bcrypt]')
run('pip install -q qdrant-client rank-bm25 httpx')
run('pip install -q FlagEmbedding torch transformers sentence-transformers')
run('pip install -q faster-whisper')
run('pip install -q pyngrok')
print('Packages installed.')

In [ ]:
# ── Step 3: Start Qdrant ───────────────────────────────────────────────────
import subprocess, time, os

run('docker pull qdrant/qdrant:v1.12.4 -q || true')
run('docker run -d --name qdrant -p 6333:6333 qdrant/qdrant:v1.12.4 || docker start qdrant')
time.sleep(5)

# Fallback: run Qdrant binary if Docker not available
import urllib.request, tarfile, pathlib
qdrant_bin = pathlib.Path('/usr/local/bin/qdrant')
if not qdrant_bin.exists():
    print('Downloading Qdrant binary...')
    url = 'https://github.com/qdrant/qdrant/releases/download/v1.12.4/qdrant-x86_64-unknown-linux-gnu.tar.gz'
    urllib.request.urlretrieve(url, '/tmp/qdrant.tar.gz')
    with tarfile.open('/tmp/qdrant.tar.gz') as t:
        t.extractall('/usr/local/bin')
    os.chmod('/usr/local/bin/qdrant', 0o755)

# Start qdrant if not running
import requests
try:
    requests.get('http://localhost:6333/healthz', timeout=3)
    print('Qdrant already running')
except:
    os.makedirs('/tmp/qdrant_storage', exist_ok=True)
    subprocess.Popen(['qdrant', '--storage-path', '/tmp/qdrant_storage'],
                     stdout=open('/tmp/qdrant.log','w'), stderr=subprocess.STDOUT)
    time.sleep(8)
    print('Qdrant started')

In [ ]:
# ── Step 4: Install Ollama and pull model ─────────────────────────────────
import subprocess, time, os, pathlib

if not pathlib.Path('/usr/local/bin/ollama').exists():
    print('Installing Ollama...')
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True)

# Start ollama server
env = os.environ.copy()
env['OLLAMA_KEEP_ALIVE'] = '5m'
subprocess.Popen(['ollama', 'serve'], env=env,
                 stdout=open('/tmp/ollama.log','w'), stderr=subprocess.STDOUT)
time.sleep(6)

MODEL = os.environ.get('LLM_MODEL', 'qwen3:4b')
print(f'Pulling {MODEL} (this may take several minutes)...')
subprocess.run(f'ollama pull {MODEL}', shell=True)
print('Ollama ready')

In [ ]:
# ── Step 5: Set up repo code ───────────────────────────────────────────────
import sys, os, pathlib

# If running from a cloned repo, adjust this path
REPO_ROOT = pathlib.Path('/kaggle/working/Hubmicroo_VoiceAssistant')

if not REPO_ROOT.exists():
    print('Cloning repo from GitHub...')
    subprocess.run(
        'git clone https://github.com/ai-with-abdullah/Hubmicroo_VoiceAssistant.git /kaggle/working/Hubmicroo_VoiceAssistant',
        shell=True
    )

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# Write .env
env_content = """
LLM_MODEL=qwen3:4b
OLLAMA_BASE_URL=http://localhost:11434
QDRANT_URL=http://localhost:6333
EMBED_MODEL=BAAI/bge-m3
EMBED_DEVICE=cuda
WHISPER_DEVICE=cuda
WHISPER_MODEL=base
TTS_ENABLED=false
DATA_DIR=/kaggle/working/Hubmicroo_VoiceAssistant/backend/data
PRODUCTS_FILE=/kaggle/working/Hubmicroo_VoiceAssistant/backend/data/products.json
PAGES_FILE=/kaggle/working/Hubmicroo_VoiceAssistant/backend/data/pages.json
ADMIN_USERNAME=admin
ADMIN_PASSWORD=kaggle-test-pass
SECRET_KEY=kaggle-testing-secret-key-not-for-prod
CORS_ORIGINS=*
"""
pathlib.Path('/kaggle/working/Hubmicroo_VoiceAssistant/.env').write_text(env_content)
print('Environment configured')

In [ ]:
# ── Step 6: Seed the index ────────────────────────────────────────────────
import sys, os
os.chdir('/kaggle/working/Hubmicroo_VoiceAssistant')

from backend.app.config import get_settings
from backend.app import store
from backend.app.indexer import rebuild_all

store.ensure_collection()
count = rebuild_all()
print(f'Index seeded: {count} items')

In [ ]:
# ── Step 7: Start FastAPI backend ─────────────────────────────────────────
import subprocess, time

server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'backend.app.main:app',
     '--host', '0.0.0.0', '--port', '8000'],
    cwd='/kaggle/working/Hubmicroo_VoiceAssistant',
    stdout=open('/tmp/fastapi.log', 'w'),
    stderr=subprocess.STDOUT
)
time.sleep(10)
print('Backend started. Checking health...')

import urllib.request, json
try:
    with urllib.request.urlopen('http://localhost:8000/api/health', timeout=15) as r:
        h = json.loads(r.read())
    print('Health:', h)
except Exception as e:
    print(f'Health check failed: {e}')
    print('Check /tmp/fastapi.log for errors')

In [ ]:
# ── Step 8: Expose via ngrok ──────────────────────────────────────────────
# Get a free ngrok token at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ''  # <-- paste your token here

from pyngrok import ngrok, conf

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url
else:
    public_url = 'http://localhost:8000 (no ngrok token — only accessible from this notebook)'

print('\n' + '='*60)
print(f'  Assistant URL : {public_url}/')
print(f'  Admin Panel   : {public_url}/admin')
print(f'  API Docs      : {public_url}/docs')
print('='*60 + '\n')

In [ ]:
# ── Step 9: Run eval harness ──────────────────────────────────────────────
import subprocess

result = subprocess.run(
    ['python', 'eval/run_eval.py',
     '--base-url', 'http://localhost:8000',
     '--out', '/tmp/eval_results.json'],
    cwd='/kaggle/working/Hubmicroo_VoiceAssistant',
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:500])

In [ ]:
# ── Step 10: Quick smoke test ─────────────────────────────────────────────
import urllib.request, json

queries = [
    ('en', 'Do you have wireless headphones under 5000?'),
    ('ur', 'kya aap ke paas wireless headphones hain?'),
    ('ar', 'هل لديكم سماعات بلوتوث؟'),
]

for lang, q in queries:
    payload = json.dumps({'message': q}).encode()
    req = urllib.request.Request(
        'http://localhost:8000/api/chat',
        data=payload,
        headers={'Content-Type': 'application/json'}
    )
    with urllib.request.urlopen(req, timeout=60) as r:
        resp = json.loads(r.read())
    print(f'[{lang.upper()}] Q: {q[:50]}')
    print(f'       A: {resp["answer"][:100]}')
    print(f'  Products: {[p["name"] for p in resp["products"]]}')
    print()